# Agent Evaluation Demo

This notebook walks through two things:

1. **Chat with your data** — use the `agent-framework` library to build a simple agent that queries a PostgreSQL product database and generate a set of question/answer samples.
2. **Evaluate the responses** — use `azure-ai-evaluation` to score each answer for **fluency** and **relevance** and **intent resolution**.

---

## Install Dependencies

In [ ]:
%pip install agent-framework azure-ai-evaluation azure-identity psycopg2-binary pandas python-dotenv --quiet

---
## Step 1: Setup and import enviornment variables

Load environment variables from a `.env` file. The required variables are:

```
AZURE_OPENAI_ENDPOINT=
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=
POSTGRES_HOST=
POSTGRES_DB=
POSTGRES_USER=
POSTGRES_PORT=5432
SSLMODE=require
```

In [1]:
import os
import asyncio
import urllib.parse
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

from azure.identity import AzureCliCredential, DefaultAzureCredential


print("Imports OK")

Imports OK


---
## Step 2: Connect to the Database

We build a connection URI using Azure AD token-based authentication (passwordless).

In [2]:
def get_connection_uri() -> str:
    dbhost = os.getenv("POSTGRES_HOST")
    dbname = os.getenv("POSTGRES_DB")
    dbuser = urllib.parse.quote(os.getenv("POSTGRES_USER"))
    sslmode = os.getenv("SSLMODE", "require")
    dbport  = os.getenv("POSTGRES_PORT", "5432")

    credential = DefaultAzureCredential()
    token = credential.get_token("https://ossrdbms-aad.database.windows.net/.default").token
    password = urllib.parse.quote_plus(token)

    return f"postgresql://{dbuser}:{password}@{dbhost}:{dbport}/{dbname}?sslmode={sslmode}"

conn_uri = get_connection_uri()
print("Connection URI retrieved successfully.")

Connection URI retrieved successfully.


---
## Step 3: Define the Database Tool

We create a plugin class with different types of tools. One specific and other ones generic.

In [4]:
import psycopg2
from typing import Optional
import pandas as pd
import json

########## GENERIC PLUGIN ##########
class PG_Plugin:

    def __init__(self, db_uri: str):
        self.db_uri = db_uri  # Store URI instead of connection
        # Test connection
        conn = psycopg2.connect(db_uri)
        conn.close()
        print("Connected to company's database successfully.")

    def _get_connection(self):
        """Create a new connection for each operation."""
        return psycopg2.connect(self.db_uri)
    
    async def get_product_info(
        self,
        product_name: Optional[str] = None,
        product_id: Optional[int] = None,
    ) -> list[dict]:
        """Gets product information (id, name, inventory, price, refurbished, category) by name or ID."""
        if not product_name and not product_id:
            return [{"error": "Provide product_name or product_id."}]

        query = """
            SELECT product_id, name, inventory, price, refurbished, category
            FROM products
            WHERE (LOWER(name) = LOWER(%(product_name)s) AND %(product_name)s IS NOT NULL)
               OR (product_id = %(product_id)s           AND %(product_id)s  IS NOT NULL)
        """
        conn = self._get_connection()
        cursor = conn.cursor()
        cursor.execute(query, {"product_name": product_name, "product_id": product_id})
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        cursor.close()
        conn.close()
        return pd.DataFrame(rows, columns=columns).to_dict(orient="records")

    async def execute_query(self, query: str) -> list:
        conn = self._get_connection()
        try:
            query_cursor = conn.cursor()
            query_cursor.execute(query)
            if query.strip().upper().startswith("SELECT"):
                result = query_cursor.fetchall()
            else:
                result = ["Only read-only SELECT queries are allowed."]
                conn.commit()     
        except psycopg2.Error as e:
            conn.rollback()
            result = [str(e)]
        finally:
            query_cursor.close()
            conn.close()
        return result

    async def get_schema_info(self) -> str:
        print("Getting schema")
        conn = self._get_connection()
        try:
            query = """
            SELECT
                cols.table_schema,
                cols.table_name,
                cols.column_name,
                cols.data_type,
                cols.is_nullable,
                cons.constraint_type,
                cons.constraint_name,
                fk.references_table AS referenced_table,
                fk.references_column AS referenced_column
            FROM information_schema.columns cols
            LEFT JOIN information_schema.key_column_usage kcu
                ON cols.table_schema = kcu.table_schema
                AND cols.table_name = kcu.table_name
                AND cols.column_name = kcu.column_name
            LEFT JOIN information_schema.table_constraints cons
                ON kcu.table_schema = cons.table_schema
                AND kcu.table_name = cons.table_name
                AND kcu.constraint_name = cons.constraint_name
            LEFT JOIN (
                SELECT
                    rc.constraint_name,
                    kcu.table_name AS references_table,
                    kcu.column_name AS references_column
                FROM information_schema.referential_constraints rc
                JOIN information_schema.key_column_usage kcu
                    ON rc.unique_constraint_name = kcu.constraint_name
            ) fk
                ON cons.constraint_name = fk.constraint_name
            WHERE cols.table_schema = 'public'
            ORDER BY cols.table_schema, cols.table_name, cols.ordinal_position;
            """
            schema_cur = conn.cursor()
            schema_cur.execute(query)
            columns = [desc[0] for desc in schema_cur.description]
            rows = schema_cur.fetchall()
            schema_cur.close()
            schema_info = [dict(zip(columns, row)) for row in rows]
            return json.dumps(schema_info, indent=2)
        finally:
            conn.close()




---
## Step 4: Create the Agents


In [5]:
from agent_framework.foundry import FoundryChatClient
from agent_framework import Agent
generic_plugin = PG_Plugin(conn_uri)
chat_client=FoundryChatClient(project_endpoint= os.getenv("AZURE_OPENAI_ENDPOINT"),
                                model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
                                credential=AzureCliCredential())
agent1 = Agent(
    client = chat_client,
    instructions="You are a read-only agent for company's database.",
    tools = [generic_plugin.get_product_info],
    name="product_info_agent")

agent2 = Agent(
    client = chat_client,
    instructions="You are a read-only agent for company's database. you can get the full database schema and then create appropriate SQL queries to answer the user's question. Always use get_schema_info tool to get the latest schema before generating SQL query. Only use execute_query tool to execute SELECT statements. Never generate any other type of statement.",
    tools = [generic_plugin.execute_query, generic_plugin.get_schema_info],
    name="product_info_agent")

Connected to company's database successfully.


---
## Step 5: Generate Q&A Samples

We run the agent against a list of sample questions and collect the responses into a dataset.

In [ ]:
# Replace with product names/IDs that actually exist in your database

async def ask(agent, question: str) -> dict:
    result = await agent.run(question)
    return {
        "question": question,
        "answer": result.text,
        "total_tokens": result.to_dict()["usage_details"]["total_token_count"],
    }

import asyncio

async def generate_samples(agent, questions: list[str]) -> list[dict]:
    tasks = [ask(agent, q) for q in questions]   # create coroutines
    samples = await asyncio.gather(*tasks)
    return samples



In [7]:
questions = [
    "What is the price of the product named 'iPhone 14 Pro'?",
    "Is product ID 6 refurbished?",
    "What category does 'Macbook Pro M3' belong to?",
    "How many units of 'Sony TV OLED' are in inventory?",
    "Compare product 'Tablet 0.2' with 'Tablet 0.2 new'. What are the differences?",
    "How many products in category 'Laptops' are available?",
    "What is the total number of products sold?",
]

In [8]:
samples_agent1 = await generate_samples(agent1,questions)


In [ ]:
samples_agent2 = await generate_samples(agent2,questions)

In [ ]:
samples_agent1

In [ ]:
samples_agent2

In [ ]:
# View the Q&A samples as a table
df_samples1 = pd.DataFrame(samples_agent1)
df_samples1[["question", "answer"]]


In [ ]:
df_samples2 = pd.DataFrame(samples_agent2)
df_samples2[["question", "answer"]]

---
## Step 6: Evaluate Responses with Azure AI Evaluation SDK

We use two AI-assisted evaluators from `azure-ai-evaluation`:

| Evaluator | What it measures | Inputs |
|---|---|---|
| `FluencyEvaluator` | How natural and grammatically correct the response is | `response` |
| `RelevanceEvaluator` | How well the response addresses the question (accuracy proxy) | `query`, `response` |
| `IntentResolutionEvaluator` | How well the system identifies and understands a user's request. This understanding includes how well it scopes the user's intent, asks questions to clarify, and reminds end users of its scope of capabilities. | `query`, `response` |

All return a score from **1 (poor) to 5 (excellent)**.

### 6a. Configure the Evaluator Model

In [14]:
# The evaluators call an Azure OpenAI model to judge each response.
# This can be the same deployment you use for the agent, or a separate one.

model_config = {
    "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
    # "api_key": os.environ["AZURE_OPENAI_API_KEY"],
    "azure_deployment": "gpt-4.1",
    "api_version": "2024-08-01-preview",
}

print("Model config ready.")

Model config ready.


### 6b. Define a function to evaluate for all metrics

In [19]:
from azure.ai.evaluation import FluencyEvaluator
from azure.ai.evaluation import RelevanceEvaluator
from azure.ai.evaluation import IntentResolutionEvaluator
import logging
logging.getLogger("azure.ai.evaluation").setLevel(logging.ERROR)

def evaluate_agent_responses(samples: list[dict], model_config: dict) -> pd.DataFrame:
    relevance_evaluator = RelevanceEvaluator(model_config)
    fluency_evaluator = FluencyEvaluator(model_config)
    intent_resolution_evaluator = IntentResolutionEvaluator(model_config)

    fluency_scores = []
    relevance_scores = []
    intent_resolution_scores = []
    for sample in samples:
        score = fluency_evaluator(response=sample["answer"])
        fluency_scores.append(score["fluency"])
        # print(f"Q: {sample['question'][:60]}")
        # print(f"   Fluency score: {score['fluency']} / 5\n")
        score = relevance_evaluator(query=sample["question"], response=sample["answer"])
        relevance_scores.append(score["relevance"])
        # print(f"Q: {sample['question'][:60]}")
        # print(f"   Relevance score: {score['relevance']} / 5\n")
        score = intent_resolution_evaluator(query=sample["question"], response=sample["answer"])
        # print(f"   Intent Resolution score: {score['intent_resolution']} / 5\n")
        intent_resolution_scores.append(score["intent_resolution"])
    df_results = pd.DataFrame(samples)
    df_results["fluency"]   = fluency_scores
    df_results["relevance"] = relevance_scores
    df_results["intent_resolution"] = intent_resolution_scores
    return df_results

In [ ]:
agent1_results = evaluate_agent_responses(samples_agent1, model_config)


In [ ]:
agent2_results = evaluate_agent_responses(samples_agent2, model_config)

---
## Step 7: Summary Results

Combine all scores into a single results table.

In [ ]:

display(agent1_results[["question", "answer", "fluency", "relevance", "intent_resolution", "total_tokens"]])

print("\n--- Averages ---")
print(f"Avg Fluency  : {agent1_results['fluency'].mean():.2f} / 5")
print(f"Avg Relevance: {agent1_results['relevance'].mean():.2f} / 5")
print(f"Avg Intent Resolution: {agent1_results['intent_resolution'].mean():.2f} / 5")

In [ ]:

display(agent2_results[["question", "answer", "fluency", "relevance", "intent_resolution", "total_tokens"]])

print("\n--- Averages ---")
print(f"Avg Fluency  : {agent2_results['fluency'].mean():.2f} / 5")
print(f"Avg Relevance: {agent2_results['relevance'].mean():.2f} / 5")
print(f"Avg Intent Resolution: {agent2_results['intent_resolution'].mean():.2f} / 5")